<a href="https://colab.research.google.com/github/weagan/Speculative-Decoding/blob/main/target_gpu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Target GPU Notebook — Gemma‑2B‑IT + FastAPI + ngrok (Colab + Kaggle)

Runs the target model (Gemma‑2B‑IT) on GPU and exposes a `/verify` endpoint.
Works on both Colab and Kaggle, with ngrok tunneling and multi‑token verification.


In [ ]:
import os, torch

def detect_env():
    if "COLAB_GPU" in os.environ:
        return "colab"
    if os.path.exists("/kaggle"):
        return "kaggle"
    return "local"

ENV = detect_env()
print("Environment:", ENV)

#assert torch.cuda.is_available(), "GPU is required. Enable GPU in Colab/Kaggle."
print("CUDA devices:", torch.cuda.device_count())

Environment: colab
CUDA devices: 0


In [ ]:
HF_TOKEN = None
NGROK_AUTH_TOKEN = None

if ENV == "colab":
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
    NGROK_AUTH_TOKEN = userdata.get("NGROK_AUTH_TOKEN")
elif ENV == "kaggle":
    from kaggle_secrets import UserSecretsClient
    usc = UserSecretsClient()
    HF_TOKEN = usc.get_secret("HF_TOKEN")
    NGROK_AUTH_TOKEN = usc.get_secret("NGROK_AUTH_TOKEN")
else:
    HF_TOKEN = os.getenv("HF_TOKEN")
    NGROK_AUTH_TOKEN = os.getenv("NGROK_AUTH_TOKEN")

assert HF_TOKEN is not None, "HF_TOKEN not set. In Colab: Settings → User data. In Kaggle: Add to Secrets."
assert NGROK_AUTH_TOKEN is not None, "NGROK_AUTH_TOKEN not set. In Colab/Kaggle: store as secret."

print("Tokens loaded.")

Tokens loaded.


In [ ]:
!pip install -q transformers fastapi uvicorn[standard] pydantic requests
print("Installed dependencies.")

Installed dependencies.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

#model_name = "google/gemma-7b-it" #with draft "google/gemma-2b-it"

#model_name = "meta-llama/Llama-2-13b-hf"
model_name = "target meta-llama/Llama-2-13b-chat-hf" # with TinyLlama/TinyLlama-1.1B-Chat-v1.0
print("Loading target model:", model_name)

tokenizer = AutoTokenizer.from_pretrained(model_name, token=HF_TOKEN)
target_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    token=HF_TOKEN,
    torch_dtype=torch.float16,
    device_map="auto"
)

target_model.eval()
print("Target model loaded on GPU(s). Device map auto.")

Loading target model: google/gemma-2b-it


config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/34.2k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/13.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Target model loaded on GPU(s). Device map auto.


In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn, threading, torch, time, logging, sys

# ── Make verify() prints visible in Jupyter ───────────────────────────────────
# uvicorn runs in a daemon thread; plain print() goes to that thread's stdout
# which Jupyter never captures. Routing through the uvicorn logger fixes this:
# uvicorn installs a StreamHandler on the root logger pointed at the main
# thread's sys.stdout, so log.info() calls surface in the cell output.
logging.basicConfig(stream=sys.stdout, level=logging.INFO,
                    format="%(message)s", force=True)
log = logging.getLogger("verify")

app = FastAPI()

class VerifyRequest(BaseModel):
    messages:   list[dict] # Changed from context_text: str
    draft_text:   str

@app.get("/")
def root():
    return {"status": "ok", "model": model_name}

@app.post("/verify")
def verify(req: VerifyRequest):
    start = time.perf_counter()

    # Reconstruct context_text from messages
    context_text = "\n".join([msg["content"] for msg in req.messages if "content" in msg])

    log.info("\n" + "─"*60)
    log.info(f"[TARGET] context_text (last 120) : {repr(context_text[-120:])}")
    log.info(f"[TARGET] draft_text              : {repr(req.draft_text)}")

    context_ids = tokenizer(context_text, return_tensors="pt",
                            add_special_tokens=True).input_ids.to(target_model.device)
    draft_ids   = tokenizer(req.draft_text,   return_tensors="pt",
                            add_special_tokens=False).input_ids.to(target_model.device)

    full = torch.cat([context_ids, draft_ids], dim=1)
    P, L = context_ids.shape[1], draft_ids.shape[1]

    log.info(f"[TARGET] context tokens : {P}  |  draft tokens : {L}")
    log.info(f"[TARGET] full decoded   : {repr(tokenizer.decode(full[0], skip_special_tokens=False)[-200:])}")

    with torch.no_grad():
        logits = target_model(full).logits          # [1, P+L, V]

    target_token_ids = logits[0, P-1:P+L-1, :].argmax(dim=-1).tolist()
    target_text      = tokenizer.decode(target_token_ids, skip_special_tokens=True)

    elapsed = time.perf_counter() - start
    tps     = L / elapsed if elapsed > 0 else float("inf")

    log.info(f"[TARGET] target_token_ids : {target_token_ids}")
    log.info(f"[TARGET] target_text      : {repr(target_text)}")
    log.info(f"[TARGET] verified {L} tokens in {elapsed:.4f}s  ({tps:.1f} tok/s)")

    return {
        "draft_text":     req.draft_text,
        "target_text":    target_text,
        "elapsed":        elapsed,
        "tokens_per_sec": tps,
    }

threading.Thread(
    target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000,
                               log_config=None),   # don't let uvicorn reset logging
    daemon=True,
).start()
print("FastAPI server started on port 8000.")

FastAPI server started on port 8000.


### Debugging Target Model Input

Let's add a print statement to the `/verify` endpoint to see what `full` input sequence the `target_model` is actually processing. This will help us understand why it might be generating `<s>` tokens.

In [ ]:
# ── ngrok tunnel (pyngrok — works on Colab & Kaggle) ─────────────────────────
# pyngrok manages the binary download + auth entirely in Python;
# no wget/unzip/shell-variable expansion needed.
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyngrok"])

import time
from pyngrok import conf, ngrok

# Set auth token in Python — avoids the broken `$SHELL_VAR` expansion bug
conf.get_default().auth_token = NGROK_AUTH_TOKEN

time.sleep(2)  # let uvicorn finish binding

tunnel = ngrok.connect(8000, "http")
PUBLIC_URL = tunnel.public_url

print("=" * 55)
print(f"  Public URL : {PUBLIC_URL}")
print(f"  Endpoint   : {PUBLIC_URL}/verify  (POST)")
print("  Share this URL with your draft notebook.")
print("=" * 55)


Started server process [5048]
Waiting for application startup.
Application startup complete.
Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
Opening tunnel named: http-8000-4e7850d3-712e-4a1e-ac4a-4792c1f40aa8
Overriding default auth token
t=2026-05-30T01:56:15+0000 lvl=info msg="no configuration paths supplied"
t=2026-05-30T01:56:15+0000 lvl=info msg="using configuration at default config path" path=/root/.config/ngrok/ngrok.yml
t=2026-05-30T01:56:15+0000 lvl=info msg="open config file" path=/root/.config/ngrok/ngrok.yml err=nil
t=2026-05-30T01:56:15+0000 lvl=info msg="FIPS 140 mode" enabled=false
t=2026-05-30T01:56:15+0000 lvl=info msg="starting web service" obj=web addr=127.0.0.1:4040 allow_hosts=[]
t=2026-05-30T01:56:16+0000 lvl=info msg="client session established" obj=tunnels.session
t=2026-05-30T01:56:16+0000 lvl=info msg="tunnel session started" obj=tunnels.session
t=2026-05-30T01:56:16+0000 lvl=info msg=start pg=/api/tunnels id=3d3a6cb940d67d26
t=2026-05-30T01:56